# 📓 Notebook 6｜最佳特徵生成：Fisher LDA（站 6）

> 對應講義 Part 6（5.8）。不只「挑」特徵，還能「造」新特徵——把高維向量
> 投影到一條線上，讓投影後兩類分最開。這條線就是 Fisher 判別方向。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# 兩類二維高斯，共變異數不同（非對角），重現課本 Figure 5.6b 的精神
mu1 = np.array([1.0, 1.0]); mu2 = np.array([-1.0, -1.0])
Sigma = np.array([[1.0, 0.8], [0.8, 1.0]])   # 非對角 → w 不平行於 μ1−μ2
X1 = rng.multivariate_normal(mu1, Sigma, 200)
X2 = rng.multivariate_normal(mu2, Sigma, 200)

### Fisher 準則與閉式解

最大化廣義 Rayleigh 商 $FDR(\boldsymbol w)=\frac{\boldsymbol w^T S_b \boldsymbol w}{\boldsymbol w^T S_w \boldsymbol w}$，
兩類時閉式解是

$$\boldsymbol w = S_w^{-1}(\boldsymbol\mu_1-\boldsymbol\mu_2)$$

In [ ]:
# 親手算 Fisher 方向
Sw = np.cov(X1.T) + np.cov(X2.T)
w = np.linalg.inv(Sw) @ (mu1 - mu2)
w = w / np.linalg.norm(w)     # 只看方向，正規化
print('Fisher 方向 w =', np.round(w, 3))
print('μ1−μ2 方向 =', np.round((mu1 - mu2) / np.linalg.norm(mu1 - mu2), 3))
print('→ 因共變異數非對角，w 不平行於 μ1−μ2（課本 Figure 5.6b）')

### 投影後看分離度

把兩類投影到 w 上，畫直方圖，看重疊程度。

In [ ]:
proj1 = X1 @ w; proj2 = X2 @ w
fdr = (proj1.mean() - proj2.mean()) ** 2 / (proj1.var() + proj2.var())
print(f'投影後 FDR = {fdr:.3f}')

plt.figure(figsize=(8, 3.5))
plt.subplot(1, 2, 1)
plt.scatter(X1[:, 0], X1[:, 1], s=8, alpha=.5, label='ω1')
plt.scatter(X2[:, 0], X2[:, 1], s=8, alpha=.5, label='ω2')
plt.plot([0, w[0] * 3], [0, w[1] * 3], 'r-', lw=2, label='Fisher w')
plt.legend(); plt.title('原始空間 + Fisher 方向')
plt.subplot(1, 2, 2)
plt.hist(proj1, bins=30, alpha=.6, label='ω1')
plt.hist(proj2, bins=30, alpha=.6, label='ω2')
plt.legend(); plt.title('投影到 w 後（分離度）')
plt.tight_layout(); plt.show()

# ✏️ 練習：把 Sigma 改成對角矩陣 [[1,0],[0,1]]，看 w 是否變平行於 μ1−μ2
